# lab 3: gan pe mnist

scopul lucrării: să construim și să antrenăm o rețea gan (generative adversarial network) care desenează cifre noi scrise de mână.

o rețea gan are două rețele care joacă una împotriva celeilalte:

* generatorul primește zgomot aleator și face o imagine falsă
* discriminatorul se uită la o imagine și spune dacă este reală sau falsă

generatorul încearcă să păcălească discriminatorul. discriminatorul încearcă să nu fie păcălit. cu timpul imaginile false încep să arate ca niște cifre reale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

## 1. datele

încărcăm mnist din keras. etichetele nu ne trebuie, gan învață doar din imagini.

aducem pixelii în intervalul de la minus 1 la 1. facem asta pentru că generatorul se termină cu tanh, care dă valori în același interval.

apoi facem din fiecare imagine un vector de 784 de valori, le amestecăm și le grupăm în loturi de câte 128.

In [ ]:
(x_train, _), (_, _) = keras.datasets.mnist.load_data()

x_train = (x_train.astype("float32") - 127.5) / 127.5
x_train = x_train.reshape(-1, 784)

print(x_train.shape, x_train.min(), x_train.max())

In [ ]:
plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
plt.show()

In [ ]:
BATCH = 128

dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(BATCH, drop_remainder=True)

## 2. generatorul

generatorul primește un vector aleator de 100 de numere și dă o imagine de 784 de pixeli.

straturile sunt 100, apoi 256, apoi 512 și la final 784. în interior folosim LeakyReLU, iar la final tanh.

In [ ]:
NOISE = 100

generator = keras.Sequential([
    layers.Input(shape=(NOISE,)),
    layers.Dense(256),
    layers.LeakyReLU(0.2),
    layers.Dense(512),
    layers.LeakyReLU(0.2),
    layers.Dense(784, activation="tanh"),
], name="generator")

generator.summary()

## 3. discriminatorul

discriminatorul primește o imagine de 784 de pixeli. el dă un singur număr: șansa ca imaginea să fie reală.

straturile sunt 784, apoi 512, apoi 256 și la final 1. folosim Dropout, ca discriminatorul să nu câștige prea repede jocul.

In [ ]:
discriminator = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512),
    layers.LeakyReLU(0.2),
    layers.Dropout(0.3),
    layers.Dense(256),
    layers.LeakyReLU(0.2),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
], name="discriminator")

discriminator.summary()

## 4. funcțiile de pierdere și optimizatorii

ambele rețele folosesc binary cross entropy.

* discriminatorul vrea să spună 1 pentru imaginile reale și 0 pentru cele false
* generatorul vrea ca discriminatorul să spună 1 pentru imaginile lui false

fiecare rețea are propriul optimizator adam.

In [ ]:
bce = keras.losses.BinaryCrossentropy()

def d_loss_fn(real_out, fake_out):
    real_loss = bce(tf.ones_like(real_out), real_out)
    fake_loss = bce(tf.zeros_like(fake_out), fake_out)
    return real_loss + fake_loss

def g_loss_fn(fake_out):
    return bce(tf.ones_like(fake_out), fake_out)

g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)

## 5. un pas de antrenare

la fiecare pas:

1. facem imagini false din zgomot
2. arătăm discriminatorului imaginile reale și pe cele false
3. calculăm pierderea pentru fiecare rețea
4. actualizăm separat discriminatorul și generatorul

mai jos este și o funcție care arată 16 cifre din același zgomot fix. așa putem vedea cum se îmbunătățesc aceleași cifre în timp.

In [ ]:
@tf.function
def train_step(real_images):
    noise = tf.random.normal([BATCH, NOISE])

    with tf.GradientTape() as g_tape, tf.GradientTape() as d_tape:
        fake_images = generator(noise, training=True)

        real_out = discriminator(real_images, training=True)
        fake_out = discriminator(fake_images, training=True)

        d_loss = d_loss_fn(real_out, fake_out)
        g_loss = g_loss_fn(fake_out)

    d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))
    g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))

    return d_loss, g_loss

In [ ]:

fixed_noise = tf.random.normal([16, NOISE])

def show_images(epoch):
    imgs = generator(fixed_noise, training=False).numpy()
    imgs = (imgs + 1) / 2 
    plt.figure(figsize=(4, 4))
    for i in range(16):
        plt.subplot(4, 4, i + 1)
        plt.imshow(imgs[i].reshape(28, 28), cmap="gray")
        plt.axis("off")
    plt.suptitle(f"epoch {epoch}")
    plt.show()

## 6. antrenarea

antrenăm 30 de epoci și salvăm pierderea medie pentru fiecare epocă.

la fiecare 10 epoci ne uităm la cifrele generate. la epoca 0 generatorul nu este antrenat, deci vedem doar zgomot.

In [ ]:
EPOCHS = 30
d_hist, g_hist = [], []

show_images(0)

for epoch in range(1, EPOCHS + 1):
    d_sum, g_sum, n = 0.0, 0.0, 0
    for batch in dataset:
        d_loss, g_loss = train_step(batch)
        d_sum += float(d_loss)
        g_sum += float(g_loss)
        n += 1

    d_hist.append(d_sum / n)
    g_hist.append(g_sum / n)
    print(f"epoch {epoch:2d}  d_loss {d_hist[-1]:.3f}  g_loss {g_hist[-1]:.3f}")

    if epoch % 10 == 0:
        show_images(epoch)

## 7. graficul pierderilor

la gan pierderile nu scad ca la o antrenare obișnuită. cele două rețele se luptă între ele, de aceea liniile rămân cam la același nivel.

asta este normal, dacă nicio pierdere nu scade până la zero.

In [ ]:
plt.plot(d_hist, label="discriminator")
plt.plot(g_hist, label="generator")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()

## 8. cifre noi

dăm generatorului zgomot aleator nou și primim cifre pe care nu le-a văzut niciodată.

In [ ]:
noise = tf.random.normal([25, NOISE])
imgs = (generator(noise, training=False).numpy() + 1) / 2

plt.figure(figsize=(6, 6))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(imgs[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
plt.suptitle("generated digits")
plt.show()

## 9. evaluarea calității generării și interpretarea rezultatelor

pierderile. în primele epoci discriminatorul este mai puternic. pierderea generatorului urcă până la aproape 1.8. apoi pierderile se apropie una de alta. la epoca 30 pierderea discriminatorului este 1.14, iar a generatorului este 1.09. asta înseamnă că jocul este echilibrat și nicio rețea nu a câștigat.

imaginile. în grila finală se recunosc multe cifre, de exemplu 1, 3, 6, 7, 9 și 0. apar cifre diferite, deci generatorul nu repetă mereu aceeași imagine.

defectele. pe fundal se văd puncte de zgomot. unele cifre au linii rupte sau forme neclare. motivul este că straturile dense nu țin cont de forma imaginii, ca straturile convoluționale.

în general calitatea este bună pentru un model atât de simplu și doar 30 de epoci.

## 10. concluzii

* gan învață să facă imagini noi fără etichete, doar prin jocul dintre generator și discriminator.
* la epoca 0 ieșirea este doar zgomot. după câteva epoci apar forme de cifre, iar apoi ele devin mai clare.
* pierderile nu scad până la zero. ele rămân echilibrate, deci ambele rețele continuă să învețe.
* un gan simplu cu straturi dense face deja cifre care se pot citi.
* un gan convoluțional (dcgan) ar face cifre mai clare.